# HandPilot-Car · Kişi bazlı CNN eğitim ve değerlendirme

Bu notebook, Drive’daki `archive.zip` veri arşivini salt okunur kullanır ve tüm çıktıları `cnn_results` altında tutar. Görseller önce MediaPipe ile el ROI’sine kırpılır; CNN yalnızca RGB görüntüyü alır. Kişi bazlı ayırma yapılır, yatay çevirme kullanılmaz ve test grubu en son bir kez değerlendirilir.

**Proje anlamları:** `gaz` ileri, `fren` durma, `sol` sola ve `sag` sağa yön demektir. `fren` geri hareket olarak ele alınmaz. Veri setinde `geri` veya `diger` bulunmuyorsa bunlar için sınıf/başarı iddiası üretilmez. Sonuçlar gerçek ölçümlerdir; genelleme veya overfitting garantisi değildir.

> Eğitim Colab GPU/CPU kaynağı tüketebilir. Hücreleri sırayla çalıştırın. Notebook Drive’daki ham arşivi değiştirmez.


In [4]:
# 1) Drive, bağımlılıklar ve sabit ayarlar
from google.colab import drive
drive.mount('/content/drive')

%pip -q install mediapipe==0.10.35 scikit-learn==1.7.2 seaborn

import os, sys, json, csv, hashlib, zipfile, shutil, random, platform, tempfile, math, time
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import tensorflow as tf
import mediapipe as mp
import cv2
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    precision_recall_fscore_support, confusion_matrix, roc_auc_score,
    roc_curve, auc, brier_score_loss, classification_report)

SEED = 20261024
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
IMG_SIZE, BATCH_SIZE, MAX_EPOCHS = 192, 32, 40
CV_REQUESTED = 5
ROI_MARGIN = 0.30
CLASSES_ORDER = ['diger', 'fren', 'gaz', 'sag', 'sol']
ZIP_PATH = Path('/content/drive/MyDrive/archive.zip')
RESULTS_ROOT = ZIP_PATH.parent / 'cnn_results'
EXTRACT_ROOT = Path('/content/handpilot_dataset') # Colab geçici alanı; kaynak arşiv değişmez.
TASK_MODEL = Path('/content/hand_landmarker.task')
TASK_MODEL_URL = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task'

if not ZIP_PATH.is_file():
    raise FileNotFoundError(f'Arşiv bulunamadı: {ZIP_PATH}. Drive > MyDrive içindeki dosya adını doğrulayın.')
if not zipfile.is_zipfile(ZIP_PATH):
    raise ValueError(f'Geçerli ZIP arşivi değil: {ZIP_PATH}')

def atomic_json(path, obj):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    json.loads(tmp.read_text(encoding='utf-8'))
    os.replace(tmp, path)

def atomic_csv(df, path, **kwargs):
    path = Path(path); tmp = path.with_name(path.name + '.tmp')
    df.to_csv(tmp, **kwargs)
    try: pd.read_csv(tmp, nrows=2)
    except pd.errors.EmptyDataError:
        if len(df.columns): raise
    os.replace(tmp, path)

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
probe = RESULTS_ROOT / '.write_probe.tmp'
probe.write_text('ok', encoding='utf-8')
assert probe.read_text(encoding='utf-8') == 'ok'
probe.unlink()

zip_stat = ZIP_PATH.stat()
RUN_CONFIG = {'zip_path': str(ZIP_PATH), 'zip_size': zip_stat.st_size,
    'zip_mtime_ns': zip_stat.st_mtime_ns, 'seed': SEED, 'image_size': IMG_SIZE,
    'batch_size': BATCH_SIZE, 'epochs': MAX_EPOCHS, 'cv_requested': CV_REQUESTED,
    'roi_margin': ROI_MARGIN, 'horizontal_flip': False,
    'class_order': CLASSES_ORDER, 'models': ['small_cnn','mobilenet_v3_small'],
    'optimizer': 'AdamW; Adam fallback', 'weight_decay': 1e-4,
    'early_stopping_patience': 7,
    'tensorflow': tf.__version__, 'mediapipe': mp.__version__}
CONFIG_HASH = hashlib.sha256(json.dumps(RUN_CONFIG, sort_keys=True).encode()).hexdigest()[:12]
EXTRACT_ROOT = Path(f'/content/handpilot_dataset_{CONFIG_HASH}')
RUN_DIR = RESULTS_ROOT / f'run_{CONFIG_HASH}'
for name in ['logs', 'checkpoints', 'folds', 'figures', 'roi_cache', 'reports']:
    (RUN_DIR / name).mkdir(parents=True, exist_ok=True)
manifest = {'config': RUN_CONFIG, 'config_hash': CONFIG_HASH, 'run_dir': str(RUN_DIR),
    'python': platform.python_version(), 'tensorflow': tf.__version__, 'mediapipe': mp.__version__,
    'numpy': np.__version__, 'opencv': cv2.__version__, 'pillow': Image.__version__,
    'created_utc': datetime.now(timezone.utc).isoformat()}
atomic_json(RUN_DIR / 'manifest.json', manifest)
print('Run:', RUN_DIR, '| TensorFlow:', tf.__version__, '| MediaPipe:', mp.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Run: /content/drive/MyDrive/cnn_results/run_f2d18ed71855 | TensorFlow: 2.21.0 | MediaPipe: 0.10.35
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2) Arşivi doğrula ve güvenli biçimde aç

Bu aşamada arşiv CRC ve ZIP içi yolları doğrulanır. Kopyalama Drive’a değil geçici Colab alanına yapılır; checkpoint ve raporlar yalnızca `cnn_results` altına yazılır. Tamamlanma işareti yoksa çıkarma yeniden başlatılır.


In [ ]:
if not EXTRACT_ROOT.exists() or not (EXTRACT_ROOT / '.archive_complete.json').exists():
    if EXTRACT_ROOT.exists(): shutil.rmtree(EXTRACT_ROOT)
    with zipfile.ZipFile(ZIP_PATH) as zf:
        bad = zf.testzip()
        if bad: raise ValueError(f'ZIP bozuk üye içeriyor: {bad}')
        root = EXTRACT_ROOT.resolve()
        for name in zf.namelist():
            target = (EXTRACT_ROOT / name).resolve()
            if target != root and root not in target.parents:
                raise ValueError(f'Güvenli olmayan ZIP yolu: {name}')
        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
        zf.extractall(EXTRACT_ROOT)
    atomic_json(EXTRACT_ROOT / '.archive_complete.json', {'zip_size': zip_stat.st_size})

status = {'stage': 'archive_verified_extracted', 'updated_utc': datetime.now(timezone.utc).isoformat()}
atomic_json(RUN_DIR / 'status.json', status)
print('Arşiv doğrulandı ve geçici alana açıldı:', EXTRACT_ROOT)


Arşiv doğrulandı ve geçici alana açıldı: /content/handpilot_dataset_f2d18ed71855


## 3) Veri denetimi: klasör etiketleri, CSV, bozuk/tekrarlı görseller

Yalnızca arşivde gerçekten bulunan sınıflar kullanılır. Klasör ile CSV çelişen veya tekil biçimde eşleşmeyen örnekler eğitimden çıkarılır ve nedenleri raporlanır. `person` alanı gerekir; aksi halde kişi bazlı sızıntısız değerlendirme yapılamaz ve notebook durur.


In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
all_files = sorted([p for p in EXTRACT_ROOT.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
csv_paths = sorted(p for p in EXTRACT_ROOT.rglob('kayit.csv') if p.is_file())
if not csv_paths: raise FileNotFoundError('ZIP içinde kayit.csv bulunamadı; kişi bazlı değerlendirme için gerekli.')
meta_path = csv_paths[0]
meta = pd.read_csv(meta_path, dtype=str).fillna('')
required = {'file', 'gesture', 'person'}
if not required.issubset(meta.columns): raise ValueError(f'kayit.csv eksik sütunlar: {required-set(meta.columns)}')

def norm_rel(s): return str(s).replace('\\','/').lstrip('./').casefold()
by_rel = {norm_rel(r['file']): r.to_dict() for _,r in meta.iterrows()}
by_base = defaultdict(list)
for _,r in meta.iterrows(): by_base[Path(norm_rel(r['file'])).name].append(r.to_dict())
folder_labels = {x.casefold() for x in CLASSES_ORDER}
accepted, rejected, seen_hash = [], [], set()
found_folder_counts = Counter()
for p in all_files:
    rel = p.relative_to(EXTRACT_ROOT).as_posix()
    parts = [x.casefold() for x in p.relative_to(EXTRACT_ROOT).parts[:-1]]
    labels = [x for x in parts if x in folder_labels]
    label = labels[-1] if labels else None
    if label is None:
        rejected.append({'file':rel,'reason':'sınıf klasörü tanınmadı'}); continue
    found_folder_counts[label] += 1
    row = by_rel.get(norm_rel(rel))
    if row is None:
        candidates = by_base.get(p.name.casefold(), [])
        row = candidates[0] if len(candidates) == 1 else None
    if row is None:
        rejected.append({'file':rel,'reason':'CSV eşleşmesi yok veya tekil değil'}); continue
    csv_label = row['gesture'].strip().casefold()
    if csv_label != label:
        rejected.append({'file':rel,'reason':f'etiket çelişkisi: klasör={label}, CSV={csv_label}'}); continue
    person = row['person'].strip()
    if not person:
        rejected.append({'file':rel,'reason':'person alanı boş'}); continue
    try:
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            width,height,fmt = im.width,im.height,im.format
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
    except Exception as e:
        rejected.append({'file':rel,'reason':'okunamayan görsel: '+str(e)[:160]}); continue
    if digest in seen_hash:
        rejected.append({'file':rel,'reason':'yinelenen SHA-256'}); continue
    seen_hash.add(digest)
    accepted.append({'file':rel,'path':str(p),'label':label,'person':person,'sha256':digest,
        'width':width,'height':height,'format':fmt,
        'session':row.get('computer_session',''),'round':row.get('round',''),
        'mirrored':row.get('mirrored','')})
df = pd.DataFrame(accepted)
if df.empty: raise ValueError('Denetimden sonra kullanılabilir görsel kalmadı.')
classes = [c for c in CLASSES_ORDER if c in set(df.label)]
if len(classes) < 2: raise ValueError(f'En az iki gerçek sınıf gerekir; bulundu: {classes}')
if df.person.nunique() < 3: raise ValueError(f'Kişi bazlı train/validation/test için en az 3 kişi gerekir; bulundu: {df.person.nunique()}')
atomic_csv(pd.DataFrame(rejected), RUN_DIR/'reports'/'rejected_records.csv', index=False)
atomic_csv(df.groupby(['label','person','session'], dropna=False).size().rename('count').reset_index(),
    RUN_DIR/'reports'/'counts_by_person_session.csv', index=False)
atomic_csv(df.groupby(['label','person','round'], dropna=False).size().rename('count').reset_index(),
    RUN_DIR/'reports'/'counts_by_person_round.csv', index=False)
manifest.update({'dataset_root':str(EXTRACT_ROOT),'csv_path':str(meta_path),'classes':classes,
    'class_counts':df.label.value_counts().to_dict(),'images_found':len(all_files),
    'accepted_before_mediapipe':len(df),'rejected_count':len(rejected),
    'rejected_reasons':dict(Counter(x['reason'].split(':')[0] for x in rejected)),
    'person_count':int(df.person.nunique()),'session_count':int(df.session.nunique()),
    'format_counts':df.format.value_counts().to_dict(),
    'dimensions':{f'{w}x{h}':int(n) for (w,h),n in df.groupby(['width','height']).size().items()},
    'mirrored_counts':df.mirrored.value_counts().to_dict()})
atomic_json(RUN_DIR/'manifest.json',manifest)
atomic_json(RUN_DIR/'status.json',{'stage':'dataset_audit_complete','accepted':len(df),'rejected':len(rejected)})
print('Sınıflar:',classes,'| kişiler:',df.person.nunique(),'| görseller:',len(df),'| reddedilen:',len(rejected))
display(df.groupby('label').agg(images=('file','size'),people=('person','nunique')))


Sınıflar: ['fren', 'gaz', 'sag', 'sol'] | kişiler: 13 | görseller: 13287 | reddedilen: 0


,images,people
label,,
fren,3397,12
gaz,3248,12
sag,3393,13
sol,3249,12


## 4) MediaPipe el algılama ve ROI önbelleği

Varsa `mp.solutions.hands` kullanılır; yoksa MediaPipe Tasks Hand Landmarker API’sine geçilir. İki API de aynı 21 noktanın sınır kutusunu çıkarır. CNN’e landmark özellikleri verilmez. Kırpma kare içinde tutulur, yön ve ayna bilgisi çevrilmez. Algılanmayan görseller çıkarılır ve sınıf/kişi bazında sayılır.


In [ ]:
# Reuse completed Drive ROI cache; stage images locally for fast randomized training reads.
from concurrent.futures import ThreadPoolExecutor, as_completed
fail_path = RUN_DIR / 'reports' / 'mediapipe_failures.csv'
if not fail_path.is_file():
    raise FileNotFoundError(f'ROI failure list missing: {fail_path}')
fail_df = pd.read_csv(fail_path, dtype=str).fillna('')
failed_files = set(fail_df['file'].astype(str))
roi_df = df.loc[~df['file'].astype(str).isin(failed_files)].copy()
local_roi_root = Path('/content/handpilot_roi_cache')
local_roi_root.mkdir(parents=True, exist_ok=True)
local_paths = [None] * len(roi_df)
items = list(enumerate(roi_df['sha256'].astype(str)))
def stage_roi(item):
    i, digest = item
    filename = digest + '.jpg'
    src = RUN_DIR / 'roi_cache' / filename
    dst = local_roi_root / filename
    if not dst.is_file(): shutil.copyfile(src, dst)
    return i, str(dst)
with ThreadPoolExecutor(max_workers=24) as pool:
    futures = [pool.submit(stage_roi, item) for item in items]
    for done, future in enumerate(as_completed(futures), 1):
        i, path = future.result(); local_paths[i] = path
        if done % 1000 == 0 or done == len(items): print(f'Yerel önbellek: {done}/{len(items)}')
roi_df['roi_path'] = local_paths
if roi_df.empty or not Path(roi_df.iloc[0]['roi_path']).is_file(): raise FileNotFoundError('Local ROI cache verification failed')
detector_api = 'cached_roi_reuse_local'
mp_fail_by_class = fail_df.groupby('label').size().to_dict() if len(fail_df) else {}
mp_fail_by_person = fail_df.groupby('person').size().to_dict() if len(fail_df) else {}
atomic_json(RUN_DIR / 'status.json', {'stage':'local_roi_cache_ready','images':len(roi_df),'failed':len(fail_df),'updated_utc':datetime.now(timezone.utc).isoformat()})
print(f'Yerel ROI hazır: {len(roi_df)} görsel; {len(fail_df)} başarısızlık.')
display(roi_df.groupby('label').size().rename('detected').to_frame())

Yerel önbellek: 1000/12458
Yerel önbellek: 2000/12458
Yerel önbellek: 3000/12458
Yerel önbellek: 4000/12458
Yerel önbellek: 5000/12458
Yerel önbellek: 6000/12458


KeyboardInterrupt: 

## 5) Kişi bazlı train/validation/test ve çapraz doğrulama

Kişi grupları test ve validation için en başta ayrılır. CV yalnızca train kişileri üzerinde `GroupKFold(groups=person)` kullanır. Bölme sınıf bulunurluğu ve gerçek örnek oranları raporlanır; split oranları kişi sayısına bağlıdır.


In [ ]:
people = np.array(sorted(roi_df.person.unique()))
gss_test=GroupShuffleSplit(n_splits=1,test_size=0.15,random_state=SEED)
dev_idx,test_idx=next(gss_test.split(roi_df,roi_df.label,groups=roi_df.person))
dev=roi_df.iloc[dev_idx].copy(); test_df=roi_df.iloc[test_idx].copy()
if dev.person.nunique()<2 or test_df.person.nunique()<1: raise ValueError('Kişi sayısı test ayırmasına yetmiyor.')
gss_val=GroupShuffleSplit(n_splits=1,test_size=0.15,random_state=SEED+1)
tr_rel,val_rel=next(gss_val.split(dev,dev.label,groups=dev.person))
train_df=dev.iloc[tr_rel].copy(); val_df=dev.iloc[val_rel].copy()
split_sets={'train':train_df,'validation':val_df,'test':test_df}
split_people={k:sorted(v.person.unique().tolist()) for k,v in split_sets.items()}
assert not(set(split_people['train'])&set(split_people['validation']))
assert not(set(split_people['train'])&set(split_people['test']))
assert not(set(split_people['validation'])&set(split_people['test']))
split_summary=[]
for name,part in split_sets.items():
    counts=part.label.value_counts().to_dict()
    missing=sorted(set(classes)-set(counts))
    split_summary.append({'split':name,'people':len(split_people[name]),'images':len(part),
        'image_fraction':len(part)/len(roi_df),'classes':counts,'missing_classes':missing})
display(pd.DataFrame([{k:v for k,v in s.items() if k!='classes'} for s in split_summary]))
atomic_json(RUN_DIR/'reports'/'person_split.json',{'seed':SEED,'people':split_people,'summary':split_summary})
if len(train_df.person.unique())<2: raise ValueError('Train kişi grupları yetersiz; CV yapılamaz.')
n_splits=min(CV_REQUESTED,len(train_df.person.unique()))
if n_splits<2: raise ValueError('En az iki train kişisi gerekli.')
cv=GroupKFold(n_splits=n_splits)
label_to_id={c:i for i,c in enumerate(classes)}
id_to_label={i:c for c,i in label_to_id.items()}
print('Sınıf sırası:',classes,'| CV fold sayısı:',n_splits)


,split,people,images,image_fraction,missing_classes
0,train,9,8497,0.682052,[]
1,validation,2,2149,0.172500,[]
2,test,2,1812,0.145449,[]


Sınıf sırası: ['fren', 'gaz', 'sag', 'sol'] | CV fold sayısı: 5


## 6) Görüntü boru hattı, modeller, checkpoint ve metrikler

Her epoch sonunda tam model checkpoint’i, geçici dosyaya yazılıp doğrulanarak güncellenir. `BackupAndRestore` optimizer durumunu kesinti sonrası geri yükler; tamamlanan CV fold’ları sonuç dosyasıyla atlanır. En iyi checkpoint ayrı tutulur. Augmentation yatay çeviri içermez.


In [17]:
AUTOTUNE=tf.data.AUTOTUNE
def decode_roi(path,label):
    raw=tf.io.read_file(path); image=tf.io.decode_jpeg(raw,channels=3)
    image=tf.image.resize(image,[IMG_SIZE,IMG_SIZE],antialias=True)
    image=tf.cast(image,tf.float32)/255.0
    return image,tf.cast(label,tf.int32)

def make_ds(part,training=False):
    labels=np.array([label_to_id[x] for x in part.label],dtype=np.int32)
    ds=tf.data.Dataset.from_tensor_slices((part.roi_path.to_numpy(),labels))
    if training: ds=ds.shuffle(len(part),seed=SEED,reshuffle_each_iteration=True)
    ds=ds.map(decode_roi,num_parallel_calls=AUTOTUNE)
    if training:
        def aug(x,y):
            x=tf.image.random_brightness(x,0.08,seed=SEED)
            x=tf.image.random_contrast(x,0.90,1.10,seed=SEED+1)
            x=tf.image.random_saturation(x,0.90,1.10,seed=SEED+2)
            return tf.clip_by_value(x,0.,1.),y
        ds=ds.map(aug,num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

def build_model(kind,finetune=False):
    inp=tf.keras.Input((IMG_SIZE,IMG_SIZE,3),name='rgb_roi')
    aug=tf.keras.Sequential([
        tf.keras.layers.RandomRotation(0.035,fill_mode='reflect'),
        tf.keras.layers.RandomTranslation(0.04,0.04,fill_mode='reflect'),
        tf.keras.layers.RandomZoom(0.08,fill_mode='reflect'),
        tf.keras.layers.RandomContrast(0.12)],name='train_augmentation')
    x=aug(inp)
    if kind=='small_cnn':
        for filters in (32,64,128):
            x=tf.keras.layers.Conv2D(filters,3,padding='same',use_bias=False,kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
            x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.Activation('relu')(x)
            x=tf.keras.layers.MaxPooling2D()(x); x=tf.keras.layers.Dropout(0.15)(x)
        x=tf.keras.layers.GlobalAveragePooling2D()(x)
    else:
        base=tf.keras.applications.MobileNetV3Small(input_shape=(IMG_SIZE,IMG_SIZE,3),include_top=False,
            weights='imagenet',include_preprocessing=False)
        base.trainable=bool(finetune)
        if finetune:
            for layer in base.layers[:-30]: layer.trainable=False
        # Keep CNN input RGB [0,1], convert to MobileNetV3 expected [-1,1].
        x=tf.keras.layers.Rescaling(scale=2.,offset=-1.)(x)
        x=base(x,training=False)
        x=tf.keras.layers.GlobalAveragePooling2D()(x)
    x=tf.keras.layers.Dropout(0.35)(x)
    out=tf.keras.layers.Dense(len(classes),activation='softmax',name='class_probabilities')(x)
    model=tf.keras.Model(inp,out,name=kind)
    lr=1e-5 if kind=='mobilenet_v3_small' and finetune else 1e-3
    try: optimizer=tf.keras.optimizers.AdamW(learning_rate=lr,weight_decay=1e-4)
    except Exception: optimizer=tf.keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=optimizer,loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    return model

class AtomicFullModelCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self,path): super().__init__(); self.path=Path(path)
    def on_epoch_end(self,epoch,logs=None):
        tmp=self.path.with_name(self.path.stem+'.tmp.keras')
        self.model.save(tmp)
        tf.keras.models.load_model(tmp,compile=True)
        os.replace(tmp,self.path)
        atomic_json(RUN_DIR/'status.json',{'stage':'training_epoch_complete',
            'stage_dir':str(self.path.parent),'completed_epoch':int(epoch+1),
            'logs':{k:float(v) for k,v in (logs or {}).items()},
            'updated_utc':datetime.now(timezone.utc).isoformat()})

class AtomicBestCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self,path):
        super().__init__(); self.path=Path(path); self.best=float('inf')
        hist=self.path.parent/'history.json'
        if hist.exists():
            try: self.best=min((r['val_loss'] for r in json.loads(hist.read_text(encoding='utf-8')) if 'val_loss' in r),default=self.best)
            except Exception: pass
    def on_epoch_end(self,epoch,logs=None):
        value=(logs or {}).get('val_loss')
        if value is None or value>=self.best: return
        self.best=float(value)
        tmp=self.path.with_name(self.path.stem+'.tmp.keras')
        self.model.save(tmp)
        tf.keras.models.load_model(tmp,compile=True)
        os.replace(tmp,self.path)

class AtomicHistoryJson(tf.keras.callbacks.Callback):
    def __init__(self,path):
        super().__init__(); self.path=Path(path); self.rows=[]
        if self.path.exists():
            try: self.rows=json.loads(self.path.read_text(encoding='utf-8'))
            except Exception: self.rows=[]
    def on_epoch_end(self,epoch,logs=None):
        self.rows.append({'epoch':int(epoch+1),**{k:float(v) for k,v in (logs or {}).items()}})
        atomic_json(self.path,self.rows)

def callbacks_for(stage_dir):
    stage_dir=Path(stage_dir); stage_dir.mkdir(parents=True,exist_ok=True)
    return [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=7,restore_best_weights=True),
        AtomicBestCheckpoint(stage_dir/'best.keras'),
        AtomicFullModelCheckpoint(stage_dir/'last.keras'),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.3,patience=3,min_lr=1e-7),
        tf.keras.callbacks.CSVLogger(str(stage_dir/'history.csv'),append=True),
        AtomicHistoryJson(stage_dir/'history.json'),
        tf.keras.callbacks.TensorBoard(log_dir=str(stage_dir/'tensorboard'),update_freq='epoch'),
        tf.keras.callbacks.TerminateOnNaN(),
        tf.keras.callbacks.BackupAndRestore(backup_dir=str(stage_dir/'backup'),save_freq='epoch')]

def class_weights(part):
    counts=part.label.value_counts().to_dict(); total=len(part); k=len(classes)
    return {label_to_id[c]:float(total/(k*counts[c])) for c in classes if counts.get(c,0)>0}

def evaluate_model(model,part,split_name,output_dir):
    y=np.array([label_to_id[c] for c in part.label],dtype=int)
    probs=model.predict(make_ds(part),verbose=0)
    pred=probs.argmax(1); present=sorted(set(y.tolist()))
    result={'split':split_name,'n':len(y),'accuracy':float(accuracy_score(y,pred)),
        'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
        'macro_f1':float(f1_score(y,pred,average='macro',zero_division=0)),
        'weighted_f1':float(f1_score(y,pred,average='weighted',zero_division=0)),
        'class_order':classes,'missing_true_classes':[classes[i] for i in range(len(classes)) if i not in present]}
    p,r,f,s=precision_recall_fscore_support(y,pred,labels=list(range(len(classes))),zero_division=0)
    result['per_class']={classes[i]:({'precision':float(p[i]),'recall':float(r[i]),'f1':float(f[i]),'support':int(s[i])}
        if s[i] > 0 else {'precision':None,'recall':None,'f1':None,'support':0,'status':'test/fold gerçek sınıf örneği yok'}) for i in range(len(classes))}
    cm=confusion_matrix(y,pred,labels=list(range(len(classes))))
    cm_norm=np.divide(cm,cm.sum(axis=1,keepdims=True),out=np.zeros_like(cm,dtype=float),where=cm.sum(axis=1,keepdims=True)!=0)
    Path(output_dir).mkdir(parents=True,exist_ok=True)
    pd.DataFrame(cm,index=classes,columns=classes).to_csv(Path(output_dir)/f'{split_name}_confusion_raw.csv')
    pd.DataFrame(cm_norm,index=classes,columns=classes).to_csv(Path(output_dir)/f'{split_name}_confusion_normalized.csv')
    fig,ax=plt.subplots(figsize=(7,6)); sns.heatmap(cm,annot=True,fmt='d',xticklabels=classes,yticklabels=classes,cmap='Blues',ax=ax)
    ax.set(xlabel='Tahmin',ylabel='Gerçek',title=f'{split_name}: ham karışıklık matrisi'); fig.tight_layout(); fig.savefig(Path(output_dir)/f'{split_name}_confusion.png'); plt.close(fig)
    if len(present)==len(classes):
        y_bin=np.eye(len(classes))[y]
        try:
            result['roc_auc_macro_ovr']=float(roc_auc_score(y_bin,probs,average='macro',multi_class='ovr'))
            result['roc_auc_micro_ovr']=float(roc_auc_score(y_bin,probs,average='micro',multi_class='ovr'))
            result['brier_multiclass_mean']=float(np.mean(np.sum((probs-y_bin)**2,axis=1)))
            fig,ax=plt.subplots(figsize=(7,6))
            for i,c in enumerate(classes):
                fpr,tpr,_=roc_curve(y_bin[:,i],probs[:,i]); class_auc=float(auc(fpr,tpr))
                result['per_class'][c]['roc_auc_ovr']=class_auc
                ax.plot(fpr,tpr,label=f'{c} AUC={class_auc:.3f}')
            ax.plot([0,1],[0,1],'k--'); ax.legend(); ax.set(xlabel='False positive rate',ylabel='True positive rate',title=f'{split_name}: ROC'); fig.tight_layout(); fig.savefig(Path(output_dir)/f'{split_name}_roc.png'); plt.close(fig)
        except Exception as e: result['roc_auc_unavailable']=str(e)
    else:
        result['roc_auc_unavailable']=f'Bölmede eksik gerçek sınıf var: {[classes[i] for i in range(len(classes)) if i not in present]}'
        result['brier_unavailable']=result['roc_auc_unavailable']
    atomic_json(Path(output_dir)/f'{split_name}_metrics.json',result)
    return result


## 7) Train kişileriyle CV: küçük CNN ve MobileNetV3Small

CV fold’ları yalnızca train kişilerini kullanır. Her fold ayrı sonuç/checkpoint klasöründedir; tamamlanmış `metrics.json` bulunan fold yeniden eğitilmez. Ön eğitimli ağırlıklar indirilemezse MobileNet seçeneği kayda alınır ve küçük CNN devam eder. Son model CV macro-F1 ortalamasıyla seçilir.


In [ ]:
cv_results=[]
model_kinds=['small_cnn','mobilenet_v3_small']
cv_people=train_df.person.to_numpy()
for kind in model_kinds:
    for fold,(fit_idx,hold_idx) in enumerate(cv.split(train_df,train_df.label,groups=cv_people),start=1):
        fold_dir=RUN_DIR/'folds'/kind/f'fold_{fold}'
        fold_dir.mkdir(parents=True,exist_ok=True)
        result_path=fold_dir/'metrics.json'
        if result_path.exists():
            old=json.loads(result_path.read_text(encoding='utf-8'))
            cv_results.append(old); print('Atlandı, tamamlanmış fold:',kind,fold); continue
        fit_part=train_df.iloc[fit_idx].copy(); hold_part=train_df.iloc[hold_idx].copy()
        if len(set(classes)-set(fit_part.label)):
            print('UYARI fold fit sınıfları eksik:',kind,fold,set(classes)-set(fit_part.label))
        try:
            model=build_model(kind)
        except Exception as e:
            if kind=='mobilenet_v3_small':
                note={'model':kind,'fold':fold,'status':'unavailable','reason':str(e)}
                atomic_json(fold_dir/'unavailable.json',note); print('MobileNet kullanılamadı:',e); continue
            raise
        tr_ds=make_ds(fit_part,training=True); va_ds=make_ds(hold_part)
        history=model.fit(tr_ds,validation_data=va_ds,epochs=MAX_EPOCHS,
            class_weight=class_weights(fit_part),callbacks=callbacks_for(fold_dir),verbose=1)
        best_path=fold_dir/'best.keras'
        if best_path.exists(): model=tf.keras.models.load_model(best_path,compile=True)
        metrics=evaluate_model(model,hold_part,f'cv_{kind}_fold_{fold}',fold_dir)
        metrics.update({'model':kind,'fold':fold,'fit_people':sorted(fit_part.person.unique()),
            'validation_people':sorted(hold_part.person.unique()),'epochs_ran':len(history.history.get('loss',[])),
            'status':'complete'})
        atomic_json(result_path,metrics); cv_results.append(metrics)
        pd.DataFrame(cv_results).to_json(RUN_DIR/'reports'/'cv_results.json',orient='records',force_ascii=False,indent=2)
        manifest['last_completed_cv']={'model':kind,'fold':fold}; atomic_json(RUN_DIR/'manifest.json',manifest)
        atomic_json(RUN_DIR/'status.json',{'stage':'cv_in_progress','model':kind,'fold':fold})
        del model; tf.keras.backend.clear_session()

cv_df=pd.DataFrame([r for r in cv_results if r.get('status')=='complete'])
if cv_df.empty: raise RuntimeError('Hiçbir CV fold’u tamamlanmadı.')
summary=cv_df.groupby('model').agg(folds=('fold','count'),macro_f1_mean=('macro_f1','mean'),
    macro_f1_std=('macro_f1','std'),balanced_accuracy_mean=('balanced_accuracy','mean'),
    weighted_f1_mean=('weighted_f1','mean')).sort_values('macro_f1_mean',ascending=False)
display(summary)
selected_model=summary.index[0]
atomic_json(RUN_DIR/'reports'/'cv_model_comparison.json',{'selected_model':selected_model,'summary':summary.reset_index().to_dict('records'),'fold_results':cv_results})
print('CV’ye göre seçilen mimari:',selected_model)


Atlandı, tamamlanmış fold: small_cnn 1
Atlandı, tamamlanmış fold: small_cnn 2
Epoch 4/40
 19/209 ━━━━━━━━━━━━━━━━━━━━ 15:35 5s/step - accuracy: 0.9328 - loss: 0.2863

KeyboardInterrupt: 

## 8) Seçilen mimariyi final train/validation ile eğit ve test grubunu bir kez aç

Test kişi grupları bu hücreye kadar model seçimi veya erken durdurma için kullanılmadı. Final aşama train kişileriyle öğrenir, validation kişileriyle callback’leri izler. Sonrasında test sonuçları yazılır ve raporlanır. MobileNet ağırlıkları indirilemezse seçili modeli otomatik olarak değiştirmek yerine açık hata üretir; CV karşılaştırmasında kullanılabilir model seçilmiş olmalıdır.


In [34]:

# ============================================================
# FINAL MODEL EGITIMI VE TEST
# ============================================================

import json
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from datetime import datetime, timezone
from sklearn.metrics import confusion_matrix

final_dir = RUN_DIR / 'checkpoints' / 'final'
final_dir.mkdir(parents=True, exist_ok=True)

(RUN_DIR / 'reports').mkdir(parents=True, exist_ok=True)
(RUN_DIR / 'figures').mkdir(parents=True, exist_ok=True)

final_metrics_path = RUN_DIR / 'reports' / 'final_test_metrics.json'
summary_file = final_dir / 'stage_summaries.json'

final_stage_summaries = (
    json.loads(summary_file.read_text(encoding='utf-8'))
    if summary_file.exists()
    else []
)

if final_metrics_path.exists():

    final_metrics = json.loads(
        final_metrics_path.read_text(encoding='utf-8')
    )

    print(
        'Final test daha önce tamamlanmış; '
        'saklanan tek test değerlendirmesi yüklendi.'
    )

else:

    # --------------------------------------------------------
    # FINAL EGITIM FONKSIYONU
    # --------------------------------------------------------

    def fit_stage(stage_name, finetune=False, epochs=MAX_EPOCHS):

        stage_dir = final_dir / stage_name
        stage_dir.mkdir(parents=True, exist_ok=True)

        stage_done = stage_dir / 'stage_complete.json'
        best_path = stage_dir / 'best.keras'

        # Tamamlanan aşamayı tekrar eğitme
        if stage_done.exists() and best_path.exists():

            saved = json.loads(
                stage_done.read_text(encoding='utf-8')
            )

            final_stage_summaries[:] = [
                x for x in final_stage_summaries
                if x.get('stage') != stage_name
            ]

            final_stage_summaries.append(saved)

            print(
                'Atlandı, tamamlanmış final aşaması:',
                stage_name
            )

            return tf.keras.models.load_model(
                best_path,
                compile=True
            )

        # Modeli oluştur
        model = build_model(
            selected_model,
            finetune=finetune
        )

        # Fine-tuning için head ağırlıklarını yükle
        if finetune:

            prev = final_dir / 'head' / 'best.keras'

            if not prev.exists():
                raise FileNotFoundError(
                    f'Head modeli bulunamadı: {prev}'
                )

            try:

                source = tf.keras.models.load_model(
                    prev,
                    compile=False
                )

                model.set_weights(source.get_weights())

                del source

                print(
                    'Head ağırlıkları fine-tuning modeline aktarıldı.'
                )

            except Exception as e:

                raise RuntimeError(
                    f'Head ağırlıkları kopyalanamadı: {e}'
                ) from e

        # Model eğitimi
        print('\nFinal eğitim aşaması:', stage_name)

        hist = model.fit(
            make_ds(train_df, training=True),
            validation_data=make_ds(val_df),
            epochs=epochs,
            class_weight=class_weights(train_df),
            callbacks=callbacks_for(stage_dir),
            verbose=1
        )

        # En iyi modeli yükle
        if best_path.exists():

            model = tf.keras.models.load_model(
                best_path,
                compile=True
            )

        # Eğitim geçmişi
        h = pd.DataFrame(hist.history)

        h.to_csv(
            stage_dir / 'history_summary.csv',
            index_label='epoch'
        )

        best_epoch = (
            int(h['val_loss'].idxmin())
            if 'val_loss' in h and len(h)
            else None
        )

        stage_summary = {
            'stage': stage_name,
            'epochs_ran': len(h),
            'best_epoch': best_epoch,
            'best_val_loss': (
                float(h.loc[best_epoch, 'val_loss'])
                if best_epoch is not None
                else None
            ),
            'train_loss_at_best': (
                float(h.loc[best_epoch, 'loss'])
                if best_epoch is not None and 'loss' in h
                else None
            ),
            'train_validation_loss_gap': (
                float(
                    h.loc[best_epoch, 'val_loss']
                    - h.loc[best_epoch, 'loss']
                )
                if best_epoch is not None and 'loss' in h
                else None
            ),
            'final_learning_rate': float(
                tf.keras.backend.get_value(
                    model.optimizer.learning_rate
                )
            )
        }

        final_stage_summaries[:] = [
            x for x in final_stage_summaries
            if x.get('stage') != stage_name
        ]

        final_stage_summaries.append(stage_summary)

        atomic_json(
            summary_file,
            final_stage_summaries
        )

        # Loss grafiği
        if {'loss', 'val_loss'}.issubset(h.columns):

            fig, ax = plt.subplots(figsize=(8, 4))

            ax.plot(h['loss'], label='train')
            ax.plot(h['val_loss'], label='validation')

            ax.legend()
            ax.set_title(stage_name + ' loss')

            fig.tight_layout()

            fig.savefig(
                RUN_DIR / 'figures' / f'{stage_name}_loss.png'
            )

            plt.close(fig)

        # Aşamanın tamamlandığını kaydet
        atomic_json(
            stage_done,
            stage_summary
        )

        atomic_json(
            RUN_DIR / 'status.json',
            {
                'stage': 'final_' + stage_name + '_complete',
                'epochs_ran': len(h)
            }
        )

        return model

    # --------------------------------------------------------
    # FINAL MODEL EGITIMI
    # --------------------------------------------------------

    try:

        if selected_model == 'mobilenet_v3_small':

            # 1. Aşama: Frozen backbone
            final_model = fit_stage(
                'head',
                finetune=False,
                epochs=MAX_EPOCHS
            )

            # 2. Aşama: Fine-tuning
            final_model = fit_stage(
                'finetune',
                finetune=True,
                epochs=max(5, MAX_EPOCHS // 2)
            )

        else:

            final_model = fit_stage(
                'small_cnn',
                finetune=False,
                epochs=MAX_EPOCHS
            )

    except Exception as e:

        raise RuntimeError(
            f'Seçilen mimari {selected_model} '
            f'final eğitimde başarısız oldu; '
            f'test kullanılmadı. Neden: {e}'
        ) from e

    # --------------------------------------------------------
    # FINAL TEST DEGERLENDIRMESI
    # --------------------------------------------------------

    print('\nFinal eğitim tamamlandı.')
    print('Test değerlendirmesi başlıyor...')

    final_metrics = evaluate_model(
        final_model,
        test_df,
        'test',
        RUN_DIR / 'reports'
    )

    # Gaz / fren karışıklık matrisi
    y_true = [
        label_to_id[c]
        for c in test_df.label
    ]

    y_pred = final_model.predict(
        make_ds(test_df),
        verbose=0
    ).argmax(axis=1)

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(len(classes)))
    )

    final_metrics.update({
        'selected_model': selected_model,

        'test_people': sorted(
            test_df.person.unique().tolist()
        ),

        'train_people': sorted(
            train_df.person.unique().tolist()
        ),

        'validation_people': sorted(
            val_df.person.unique().tolist()
        ),

        'gas_fren_confusion': {

            'gaz_as_fren': (
                int(cm[
                    label_to_id['gaz'],
                    label_to_id['fren']
                ])
                if {'gaz', 'fren'}.issubset(classes)
                else None
            ),

            'fren_as_gaz': (
                int(cm[
                    label_to_id['fren'],
                    label_to_id['gaz']
                ])
                if {'gaz', 'fren'}.issubset(classes)
                else None
            )
        }
    })

    # Test sonuçlarını kaydet
    atomic_json(
        final_metrics_path,
        final_metrics
    )

    # Final modeli kaydet
    final_model.save(
        final_dir / 'final_model.keras'
    )

    atomic_json(
        RUN_DIR / 'status.json',
        {
            'stage': 'final_test_complete',
            'updated_utc': datetime.now(
                timezone.utc
            ).isoformat()
        }
    )

    del final_model

# ------------------------------------------------------------
# SONUCLARI GOSTER
# ------------------------------------------------------------

print('\nFINAL SONUCLAR')

print(
    json.dumps(
        final_metrics,
        ensure_ascii=False,
        indent=2
    )
)



/usr/local/lib/python3.13/dist-packages/keras/src/applications/mobilenet_v3.py:454: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(



Final eğitim aşaması: head
Epoch 1/40
266/266 ━━━━━━━━━━━━━━━━━━━━ 47s 148ms/step - accuracy: 0.8185 - loss: 0.5400 - val_accuracy: 0.8371 - val_loss: 0.4191 - learning_rate: 0.0010
Epoch 2/40
266/266 ━━━━━━━━━━━━━━━━━━━━ 37s 140ms/step - accuracy: 0.9751 - loss: 0.1389 - val_accuracy: 0.8553 - val_loss: 0.3444 - learning_rate: 0.0010
Epoch 3/40
266/266 ━━━━━━━━━━━━━━━━━━━━ 40s 136ms/step - accuracy: 0.9816 - loss: 0.0929 - val_accuracy: 0.8255 - val_loss: 0.3942 - learning_rate: 0.0010
Epoch 4/40
266/266 ━━━━━━━━━━━━━━━━━━━━ 41s 154ms/step - accuracy: 0.9887 - loss: 0.0660 - val_accuracy: 0.8320 - val_loss: 0.3698 - learning_rate: 0.0010
Epoch 5/40
266/266 ━━━━━━━━━━━━━━━━━━━━ 37s 137ms/step - accuracy: 0.9878 - loss: 0.0560 - val_accuracy: 0.8525 - val_loss: 0.3287 - learning_rate: 0.0010
Epoch 6/40
266/266 ━━━━━━━━━━━━━━━━━━━━ 45s 154ms/step - accuracy: 0.9900 - loss: 0.0503 - val_accuracy: 0.8492 - val_loss: 0.3369 - learning_rate: 0.0010
Epoch 7/40
266/266 ━━━━━━━━━━━━━━━━━━━━ 38

/usr/local/lib/python3.13/dist-packages/keras/src/applications/mobilenet_v3.py:454: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


Head ağırlıkları fine-tuning modeline aktarıldı.

Final eğitim aşaması: finetune
Epoch 1/20
266/266 ━━━━━━━━━━━━━━━━━━━━ 53s 168ms/step - accuracy: 0.9572 - loss: 0.1328 - val_accuracy: 0.9577 - val_loss: 0.1030 - learning_rate: 1.0000e-05
Epoch 2/20
266/266 ━━━━━━━━━━━━━━━━━━━━ 39s 148ms/step - accuracy: 0.9816 - loss: 0.0512 - val_accuracy: 0.9772 - val_loss: 0.0651 - learning_rate: 1.0000e-05
Epoch 3/20
266/266 ━━━━━━━━━━━━━━━━━━━━ 41s 148ms/step - accuracy: 0.9858 - loss: 0.0428 - val_accuracy: 0.9837 - val_loss: 0.0470 - learning_rate: 1.0000e-05
Epoch 4/20
266/266 ━━━━━━━━━━━━━━━━━━━━ 46s 169ms/step - accuracy: 0.9903 - loss: 0.0306 - val_accuracy: 0.9874 - val_loss: 0.0373 - learning_rate: 1.0000e-05
Epoch 5/20
266/266 ━━━━━━━━━━━━━━━━━━━━ 44s 165ms/step - accuracy: 0.9927 - loss: 0.0227 - val_accuracy: 0.9884 - val_loss: 0.0353 - learning_rate: 1.0000e-05
Epoch 6/20
266/266 ━━━━━━━━━━━━━━━━━━━━ 39s 148ms/step - accuracy: 0.9912 - loss: 0.0246 - val_accuracy: 0.9888 - val_loss: 

## 9) Sonuç özeti

Otomatik özet yalnızca kaydedilmiş ölçümleri kullanır. Kişi bazlı oranlar görüntü oranlarından farklı olabilir. `fren` stop sınıfıdır; arşivde ayrı ve doğru etiketlenmiş geri sınıfı yoksa model dört yönlü araç modeli değildir. `diger` yoksa bilinmeyen hareketleri güvenilir reddettiği iddia edilmez.


In [ ]:
cv_summary=json.loads((RUN_DIR/'reports'/'cv_model_comparison.json').read_text(encoding='utf-8'))
report={
 'selected_model':selected_model,
 'class_order':classes,
 'gesture_semantics':{'gaz':'ileri','fren':'durma','sol':'sola yön','sag':'sağa yön'},
 'separate_reverse_class_present':'geri' in classes,
 'unknown_class_present':'diger' in classes,
 'split_people':split_people,
 'split_summary':split_summary,
 'cv_model_comparison':cv_summary['summary'],
 'test_metrics':final_metrics,
 'final_training_stages':final_stage_summaries,
 'mediapipe':{'version':mp.__version__,'api':detector_api,'detected':len(roi_df),'failed':len(fail_df),
     'failure_by_class':mp_fail_by_class,'failure_by_person':mp_fail_by_person},
 'overfitting_note':'Erken durdurma ve train/validation loss eğrileri kontrol edildi; genelleme veya overfitting yokluğu garanti edilmez.',
 'limitations':[]}
if 'geri' not in classes: report['limitations'].append('Ayrı geri etiketi yok; dört yönlü model değildir. fren durma anlamındadır.')
if 'diger' not in classes: report['limitations'].append('diger/unknown örneği yok; bilinmeyen hareket reddi ölçülmemiştir.')
if final_metrics.get('missing_true_classes'): report['limitations'].append('Test split bazı sınıfları içermiyor; sınıf başı/AUC yorumları sınırlıdır.')
atomic_json(RUN_DIR/'reports'/'final_report.json',report)
manifest.update({'selected_model':selected_model,'final_test_metrics':str(final_metrics_path),
    'completed_utc':datetime.now(timezone.utc).isoformat()})
atomic_json(RUN_DIR/'manifest.json',manifest)
display(pd.DataFrame([{'model':r['model'],'fold':r['fold'],'macro_f1':r['macro_f1'],'balanced_accuracy':r['balanced_accuracy']} for r in cv_results if r.get('status')=='complete']))
print('Final model:',selected_model)
print('Test:',{k:final_metrics.get(k) for k in ['accuracy','balanced_accuracy','macro_f1','weighted_f1','roc_auc_macro_ovr','roc_auc_micro_ovr']})
print('Gaz-fren karışması:',final_metrics.get('gas_fren_confusion'))
print('En düşük sınıf recall:',min(final_metrics['per_class'].items(),key=lambda kv:kv[1]['recall']) if final_metrics.get('per_class') else 'hesaplanamadı')
print('Kısıtlar:',report['limitations'])
print('Rapor klasörü:',RUN_DIR/'reports')


## 10) Tek görsel için tahmin yardımcı fonksiyonu

Tahmin sırasında aynı MediaPipe tespiti, kare içi ROI, RGB renk düzeni, yeniden boyutlandırma ve `[0,1]` normalizasyonu kullanılır. Fonksiyon yalnızca eğitimde bulunan sınıflar arasında olasılık döndürür; veri setinde olmayan hareketleri güvenilir biçimde reddetme özelliği sağlamaz.


In [ ]:
def predict_image(image_path):
    model_path=RUN_DIR/'checkpoints'/'final'/'final_model.keras'
    if not model_path.exists(): raise FileNotFoundError('Önce final model eğitimi tamamlanmalı.')
    global detector_api, detector
    detector_api,detector=create_detector()
    try:
        with Image.open(image_path) as im: rgb=np.asarray(im.convert('RGB'))
        box=detect_box(rgb)
        if box is None: raise ValueError('MediaPipe görüntüde el bulamadı; tahmin üretilmedi.')
        roi=rgb[box[1]:box[3],box[0]:box[2]]
        roi=tf.image.resize(roi,[IMG_SIZE,IMG_SIZE],antialias=True)
        roi=tf.cast(roi,tf.float32)/255.0
        model=tf.keras.models.load_model(model_path,compile=False)
        probs=model.predict(tf.expand_dims(roi,0),verbose=0)[0]
        return {classes[i]:float(probs[i]) for i in np.argsort(probs)[::-1]}
    finally:
        detector.close()

# Örnek kullanımı: predict_image('/content/drive/MyDrive/ornek.jpg')


In [5]:
from pathlib import Path
import json

print("HANDPILOT CNN KONTROL")

print("ZIP:", ZIP_PATH.exists())
print("ROI:", (RUN_DIR / "roi_cache").exists())

for model in ["small_cnn", "mobilenet_v3_small"]:
    folder = RUN_DIR / "folds" / model
    completed = list(folder.glob("fold_*/metrics.json"))
    print(model, "tamamlanan fold:", len(completed))

final_path = RUN_DIR / "checkpoints" / "final" / "final_model.keras"

print("Final model:", final_path.exists())

status_path = RUN_DIR / "status.json"

if status_path.exists():
    status = json.loads(status_path.read_text(encoding="utf-8"))
    print("Son aşama:", status.get("stage"))

HANDPILOT CNN KONTROL
ZIP: True
ROI: True
small_cnn tamamlanan fold: 2
mobilenet_v3_small tamamlanan fold: 0
Final model: False
Son aşama: dataset_audit_complete


In [6]:
from pathlib import Path
import json

print("CNN EĞİTİM DOSYALARI")

for model_name in ["small_cnn", "mobilenet_v3_small"]:
    print("\nMODEL:", model_name)

    for fold in range(1, 6):
        fold_dir = RUN_DIR / "folds" / model_name / f"fold_{fold}"

        metrics = fold_dir / "metrics.json"
        best = fold_dir / "best.keras"
        last = fold_dir / "last.keras"

        print(
            f"Fold {fold}:",
            "Tamamlandı" if metrics.exists() else "Tamamlanmadı",
            "| Best:", best.exists(),
            "| Last:", last.exists()
        )

CNN EĞİTİM DOSYALARI

MODEL: small_cnn
Fold 1: Tamamlandı | Best: True | Last: True
Fold 2: Tamamlandı | Best: True | Last: True
Fold 3: Tamamlanmadı | Best: True | Last: True
Fold 4: Tamamlanmadı | Best: False | Last: False
Fold 5: Tamamlanmadı | Best: False | Last: False

MODEL: mobilenet_v3_small
Fold 1: Tamamlanmadı | Best: False | Last: False
Fold 2: Tamamlanmadı | Best: False | Last: False
Fold 3: Tamamlanmadı | Best: False | Last: False
Fold 4: Tamamlanmadı | Best: False | Last: False
Fold 5: Tamamlanmadı | Best: False | Last: False


In [7]:
import json
from pathlib import Path

fold_dir = RUN_DIR / "folds" / "small_cnn" / "fold_3"

print("SMALL CNN - FOLD 3 DURUMU")
print("-----------------------------")

history_path = fold_dir / "history.json"
backup_path = fold_dir / "backup"
best_path = fold_dir / "best.keras"
last_path = fold_dir / "last.keras"

print("Best model:", best_path.exists())
print("Last model:", last_path.exists())
print("Backup klasörü:", backup_path.exists())

if history_path.exists():
    with open(history_path, encoding="utf-8") as f:
        history = json.load(f)

    print("Kaydedilen epoch sayısı:", len(history))

    if history:
        print("Son epoch:", history[-1].get("epoch"))
        print("Son train loss:", history[-1].get("loss"))
        print("Son validation loss:", history[-1].get("val_loss"))
        print("Son validation accuracy:", history[-1].get("val_accuracy"))
else:
    print("History dosyası bulunamadı.")

print("\nBackup içeriği:")

if backup_path.exists():
    for file in backup_path.rglob("*"):
        if file.is_file():
            print(file.relative_to(backup_path))

SMALL CNN - FOLD 3 DURUMU
-----------------------------
Best model: True
Last model: True
Backup klasörü: True
Kaydedilen epoch sayısı: 3
Son epoch: 3
Son train loss: 0.3612082004547119
Son validation loss: 1.490083932876587
Son validation accuracy: 0.3875545859336853

Backup içeriği:
training_metadata.json
latest.weights.h5


In [8]:
# SMALL CNN - FOLD 3 EĞİTİMİNE DEVAM

import tensorflow as tf
from pathlib import Path

fold = 3
kind = "small_cnn"

fold_dir = RUN_DIR / "folds" / kind / f"fold_{fold}"

# Fold 3 için aynı kişi bazlı veri ayrımını oluştur
cv_splits = list(
    cv.split(
        train_df,
        train_df.label,
        groups=train_df.person.to_numpy()
    )
)

fit_idx, hold_idx = cv_splits[fold - 1]

fit_part = train_df.iloc[fit_idx].copy()
hold_part = train_df.iloc[hold_idx].copy()

print("Fold:", fold)
print("Eğitim görsel sayısı:", len(fit_part))
print("Doğrulama görsel sayısı:", len(hold_part))

# Önceki eğitimle aynı mimariyi oluştur
model = build_model("small_cnn")

# Veri setlerini hazırla
train_dataset = make_ds(fit_part, training=True)
val_dataset = make_ds(hold_part, training=False)

print("\nFold 3 eğitimi devam ediyor...")

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=MAX_EPOCHS,
    class_weight=class_weights(fit_part),
    callbacks=callbacks_for(fold_dir),
    verbose=1
)

print("\nFold 3 eğitim işlemi sona erdi.")

NameError: name 'cv' is not defined

In [9]:
from sklearn.model_selection import GroupKFold

# Önce gerekli değişkenlerin bellekte olup olmadığını kontrol et
required = ["train_df", "classes", "RUN_DIR", "build_model",
            "make_ds", "callbacks_for", "class_weights"]

missing = [name for name in required if name not in globals()]

if missing:
    print("Eksik değişkenler:", missing)
    print("Önce bu değişkenleri oluşturan önceki Colab hücreleri çalıştırılmalı.")
else:
    n_splits = min(CV_REQUESTED, train_df.person.nunique())
    cv = GroupKFold(n_splits=n_splits)

    print("CV başarıyla oluşturuldu!")
    print("Fold sayısı:", n_splits)
    print("Eğitimdeki kişi sayısı:", train_df.person.nunique())

Eksik değişkenler: ['train_df', 'build_model', 'make_ds', 'callbacks_for', 'class_weights']
Önce bu değişkenleri oluşturan önceki Colab hücreleri çalıştırılmalı.


In [10]:
# Eksik değişkenleri ve ROI kayıtlarını kontrol et

required = [
    "df",
    "roi_df",
    "train_df",
    "val_df",
    "test_df",
    "cv",
    "build_model",
    "make_ds",
    "callbacks_for",
    "class_weights",
]

for name in required:
    print(f"{name}: {'HAZIR' if name in globals() else 'EKSİK'}")

print("\nKaydedilmiş Fold 3 yedeği:")
fold_dir = RUN_DIR / "folds" / "small_cnn" / "fold_3"

for filename in [
    "best.keras",
    "last.keras",
    "history.json",
    "backup/training_metadata.json",
    "backup/latest.weights.h5",
]:
    path = fold_dir / filename
    print(f"{filename}: {'VAR' if path.exists() else 'YOK'}")

df: HAZIR
roi_df: EKSİK
train_df: EKSİK
val_df: EKSİK
test_df: EKSİK
cv: EKSİK
build_model: EKSİK
make_ds: EKSİK
callbacks_for: EKSİK
class_weights: EKSİK

Kaydedilmiş Fold 3 yedeği:
best.keras: VAR
last.keras: VAR
history.json: VAR
backup/training_metadata.json: VAR
backup/latest.weights.h5: VAR


In [11]:
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
import shutil

# Önceden kaydedilen MediaPipe ROI dosyalarını kullan
fail_path = RUN_DIR / "reports" / "mediapipe_failures.csv"
roi_cache = RUN_DIR / "roi_cache"
local_roi_root = Path("/content/handpilot_roi_cache")

if not fail_path.exists():
    raise FileNotFoundError("MediaPipe başarısızlık dosyası bulunamadı.")

fail_df = pd.read_csv(fail_path, dtype=str).fillna("")

failed_files = set(fail_df["file"].astype(str))

roi_df = df.loc[
    ~df["file"].astype(str).isin(failed_files)
].copy()

local_roi_root.mkdir(parents=True, exist_ok=True)

def restore_roi(digest):
    filename = str(digest) + ".jpg"
    source = roi_cache / filename
    target = local_roi_root / filename

    if not source.exists():
        raise FileNotFoundError(f"ROI eksik: {source}")

    if not target.exists():
        shutil.copy2(source, target)

    return str(target)

with ThreadPoolExecutor(max_workers=8) as executor:
    roi_df["roi_path"] = list(
        executor.map(restore_roi, roi_df["sha256"].astype(str))
    )

print("ROI verileri hazır!")
print("Toplam görsel:", len(roi_df))
print("Kişi sayısı:", roi_df["person"].nunique())

display(roi_df.groupby("label").size())

ROI verileri hazır!
Toplam görsel: 12458
Kişi sayısı: 13


,0
label,
fren,3391
gaz,2870
sag,3133
sol,3064


In [12]:
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

gss_test = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=SEED
)

dev_idx, test_idx = next(
    gss_test.split(
        roi_df,
        roi_df.label,
        groups=roi_df.person
    )
)

dev = roi_df.iloc[dev_idx].copy()
test_df = roi_df.iloc[test_idx].copy()

gss_val = GroupShuffleSplit(
    n_splits=1,
    test_size=0.15,
    random_state=SEED + 1
)

tr_idx, val_idx = next(
    gss_val.split(
        dev,
        dev.label,
        groups=dev.person
    )
)

train_df = dev.iloc[tr_idx].copy()
val_df = dev.iloc[val_idx].copy()

classes = [c for c in CLASSES_ORDER if c in set(df.label)]

label_to_id = {c: i for i, c in enumerate(classes)}
id_to_label = {i: c for c, i in label_to_id.items()}

n_splits = min(CV_REQUESTED, train_df.person.nunique())
cv = GroupKFold(n_splits=n_splits)

print("VERİ AYRIMI HAZIR")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("CV fold:", n_splits)

VERİ AYRIMI HAZIR
Train: 8497
Validation: 2149
Test: 1812
CV fold: 5


In [13]:
import json

split_file = RUN_DIR / "reports" / "person_split.json"

if not split_file.exists():
    raise FileNotFoundError("Önceki kişi ayrımı kaydı bulunamadı.")

saved = json.loads(split_file.read_text(encoding="utf-8"))

current = {
    "train": sorted(train_df.person.unique().tolist()),
    "validation": sorted(val_df.person.unique().tolist()),
    "test": sorted(test_df.person.unique().tolist())
}

for name in current:
    match = current[name] == sorted(saved["people"][name])
    print(name, "EŞLEŞTİ" if match else "EŞLEŞMEDİ")

assert all(
    current[name] == sorted(saved["people"][name])
    for name in current
), "Veri ayrımı değişmiş! Eğitimi başlatma."

print("\nÖnceki kişi ayrımı başarıyla doğrulandı.")

train EŞLEŞTİ
validation EŞLEŞTİ
test EŞLEŞTİ

Önceki kişi ayrımı başarıyla doğrulandı.


In [18]:
required = [
    "make_ds",
    "build_model",
    "callbacks_for",
    "class_weights",
    "evaluate_model"
]

for name in required:
    print(name, "HAZIR" if name in globals() else "EKSİK")

make_ds HAZIR
build_model HAZIR
callbacks_for HAZIR
class_weights HAZIR
evaluate_model HAZIR


In [19]:
import tensorflow as tf
import json

# Fold 3'ün önceki kişi bazlı ayrımını oluştur
fold = 3
kind = "small_cnn"

fold_dir = RUN_DIR / "folds" / kind / f"fold_{fold}"

cv_splits = list(
    cv.split(
        train_df,
        train_df.label,
        groups=train_df.person
    )
)

fit_idx, hold_idx = cv_splits[fold - 1]

fit_part = train_df.iloc[fit_idx].copy()
hold_part = train_df.iloc[hold_idx].copy()

# Eğitim yedeğini kontrol et
backup_dir = fold_dir / "backup"

assert (backup_dir / "training_metadata.json").exists()
assert (backup_dir / "latest.weights.h5").exists()

# Kaydedilmiş epoch sayısını öğren
history_path = fold_dir / "history.json"

with open(history_path, encoding="utf-8") as f:
    previous_history = json.load(f)

completed_epochs = len(previous_history)

print("Fold:", fold)
print("Önceden kaydedilmiş epoch:", completed_epochs)
print("Eğitim görseli:", len(fit_part))
print("Doğrulama görseli:", len(hold_part))

# Önceki eğitimle aynı mimari
model = build_model(kind)

# Veri setleri
train_dataset = make_ds(fit_part, training=True)
val_dataset = make_ds(hold_part)

# BackupAndRestore mevcut yedeği kullanarak devam etmeyi dener.
# initial_epoch belirtmiyoruz; callback yedekteki epoch bilgisini yönetir.
print("\nFold 3 eğitimi başlıyor...")

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=MAX_EPOCHS,
    class_weight=class_weights(fit_part),
    callbacks=callbacks_for(fold_dir),
    verbose=1
)

print("\nFold 3 eğitimi tamamlandı.")

Fold: 3
Önceden kaydedilmiş epoch: 3
Eğitim görseli: 6665
Doğrulama görseli: 1832

Fold 3 eğitimi başlıyor...
Epoch 4/40
209/209 ━━━━━━━━━━━━━━━━━━━━ 47s 175ms/step - accuracy: 0.9356 - loss: 0.2582 - val_accuracy: 0.6430 - val_loss: 0.8661 - learning_rate: 0.0010
Epoch 5/40
209/209 ━━━━━━━━━━━━━━━━━━━━ 34s 164ms/step - accuracy: 0.9551 - loss: 0.1932 - val_accuracy: 0.4612 - val_loss: 1.5392 - learning_rate: 0.0010
Epoch 6/40
209/209 ━━━━━━━━━━━━━━━━━━━━ 35s 168ms/step - accuracy: 0.9616 - loss: 0.1556 - val_accuracy: 0.5912 - val_loss: 1.3075 - learning_rate: 0.0010
Epoch 7/40
209/209 ━━━━━━━━━━━━━━━━━━━━ 32s 152ms/step - accuracy: 0.9695 - loss: 0.1360 - val_accuracy: 0.4591 - val_loss: 2.0293 - learning_rate: 0.0010
Epoch 8/40
209/209 ━━━━━━━━━━━━━━━━━━━━ 33s 155ms/step - accuracy: 0.9841 - loss: 0.0918 - val_accuracy: 0.7102 - val_loss: 1.1051 - learning_rate: 3.0000e-04
Epoch 9/40
209/209 ━━━━━━━━━━━━━━━━━━━━ 45s 173ms/step - accuracy: 0.9868 - loss: 0.0835 - val_accuracy: 0.6545

In [20]:
# Small CNN - Fold 3 değerlendirme ve kayıt

import json
import tensorflow as tf

best_path = fold_dir / "best.keras"

# En iyi doğrulama kaybına sahip modeli yükle
best_model = tf.keras.models.load_model(
    best_path,
    compile=True
)

# Fold 3 doğrulama sonuçlarını hesapla
metrics = evaluate_model(
    best_model,
    hold_part,
    "cv_small_cnn_fold_3",
    fold_dir
)

# Tamamlanan fold bilgisini kaydet
metrics.update({
    "model": "small_cnn",
    "fold": 3,
    "fit_people": sorted(fit_part.person.unique().tolist()),
    "validation_people": sorted(hold_part.person.unique().tolist()),
    "epochs_ran": len(json.loads(
        (fold_dir / "history.json").read_text(encoding="utf-8")
    )),
    "status": "complete"
})

atomic_json(fold_dir / "metrics.json", metrics)

print("FOLD 3 TAMAMLANDI VE KAYDEDİLDİ")
print("Accuracy:", round(metrics["accuracy"] * 100, 2), "%")
print("Macro F1:", round(metrics["macro_f1"], 4))
print("Balanced Accuracy:", round(metrics["balanced_accuracy"], 4))
print("Kayıt:", fold_dir / "metrics.json")

FOLD 3 TAMAMLANDI VE KAYDEDİLDİ
Accuracy: 65.07 %
Macro F1: 0.6488
Balanced Accuracy: 0.6438
Kayıt: /content/drive/MyDrive/cnn_results/run_f2d18ed71855/folds/small_cnn/fold_3/metrics.json


In [21]:
# SMALL CNN - FOLD 4

kind = "small_cnn"
fold = 4

fold_dir = RUN_DIR / "folds" / kind / f"fold_{fold}"
fold_dir.mkdir(parents=True, exist_ok=True)

# Daha önce tamamlanıp tamamlanmadığını kontrol et
result_path = fold_dir / "metrics.json"

if result_path.exists():
    print("Fold 4 zaten tamamlanmış. Yeniden eğitilmeyecek.")
else:
    # Kişi bazlı Fold 4 ayrımı
    cv_splits = list(
        cv.split(
            train_df,
            train_df.label,
            groups=train_df.person
        )
    )

    fit_idx, hold_idx = cv_splits[fold - 1]

    fit_part = train_df.iloc[fit_idx].copy()
    hold_part = train_df.iloc[hold_idx].copy()

    print("Fold:", fold)
    print("Eğitim görseli:", len(fit_part))
    print("Doğrulama görseli:", len(hold_part))

    # CNN modelini oluştur
    model = build_model(kind)

    # Eğit
    history = model.fit(
        make_ds(fit_part, training=True),
        validation_data=make_ds(hold_part),
        epochs=MAX_EPOCHS,
        class_weight=class_weights(fit_part),
        callbacks=callbacks_for(fold_dir),
        verbose=1
    )

    # En iyi modeli değerlendir
    best_model = tf.keras.models.load_model(
        fold_dir / "best.keras",
        compile=True
    )

    metrics = evaluate_model(
        best_model,
        hold_part,
        "cv_small_cnn_fold_4",
        fold_dir
    )

    metrics.update({
        "model": kind,
        "fold": fold,
        "fit_people": sorted(fit_part.person.unique().tolist()),
        "validation_people": sorted(hold_part.person.unique().tolist()),
        "epochs_ran": len(history.history.get("loss", [])),
        "status": "complete"
    })

    atomic_json(result_path, metrics)

    print("\nFOLD 4 TAMAMLANDI VE KAYDEDİLDİ")
    print("Accuracy:", round(metrics["accuracy"] * 100, 2), "%")
    print("Macro F1:", round(metrics["macro_f1"], 4))
    print("Balanced Accuracy:", round(metrics["balanced_accuracy"], 4))

Fold: 4
Eğitim görseli: 6453
Doğrulama görseli: 2044
Epoch 1/40
202/202 ━━━━━━━━━━━━━━━━━━━━ 35s 149ms/step - accuracy: 0.5269 - loss: 1.1067 - val_accuracy: 0.2950 - val_loss: 1.7343 - learning_rate: 0.0010
Epoch 2/40
202/202 ━━━━━━━━━━━━━━━━━━━━ 29s 145ms/step - accuracy: 0.7979 - loss: 0.6005 - val_accuracy: 0.2970 - val_loss: 1.6903 - learning_rate: 0.0010
Epoch 3/40
202/202 ━━━━━━━━━━━━━━━━━━━━ 34s 170ms/step - accuracy: 0.8844 - loss: 0.3994 - val_accuracy: 0.5665 - val_loss: 1.0889 - learning_rate: 0.0010
Epoch 4/40
202/202 ━━━━━━━━━━━━━━━━━━━━ 35s 173ms/step - accuracy: 0.9236 - loss: 0.2874 - val_accuracy: 0.8875 - val_loss: 0.3433 - learning_rate: 0.0010
Epoch 5/40
202/202 ━━━━━━━━━━━━━━━━━━━━ 29s 144ms/step - accuracy: 0.9393 - loss: 0.2304 - val_accuracy: 0.5484 - val_loss: 1.3697 - learning_rate: 0.0010
Epoch 6/40
202/202 ━━━━━━━━━━━━━━━━━━━━ 34s 170ms/step - accuracy: 0.9475 - loss: 0.2039 - val_accuracy: 0.9256 - val_loss: 0.2884 - learning_rate: 0.0010
Epoch 7/40
202/20

In [22]:
# SMALL CNN - FOLD 5

kind = "small_cnn"
fold = 5

fold_dir = RUN_DIR / "folds" / kind / f"fold_{fold}"
fold_dir.mkdir(parents=True, exist_ok=True)

result_path = fold_dir / "metrics.json"

if result_path.exists():
    print("Fold 5 zaten tamamlanmış.")
else:
    cv_splits = list(
        cv.split(
            train_df,
            train_df.label,
            groups=train_df.person
        )
    )

    fit_idx, hold_idx = cv_splits[fold - 1]

    fit_part = train_df.iloc[fit_idx].copy()
    hold_part = train_df.iloc[hold_idx].copy()

    print("Fold:", fold)
    print("Eğitim görseli:", len(fit_part))
    print("Doğrulama görseli:", len(hold_part))

    model = build_model(kind)

    history = model.fit(
        make_ds(fit_part, training=True),
        validation_data=make_ds(hold_part),
        epochs=MAX_EPOCHS,
        class_weight=class_weights(fit_part),
        callbacks=callbacks_for(fold_dir),
        verbose=1
    )

    best_model = tf.keras.models.load_model(
        fold_dir / "best.keras",
        compile=True
    )

    metrics = evaluate_model(
        best_model,
        hold_part,
        "cv_small_cnn_fold_5",
        fold_dir
    )

    metrics.update({
        "model": kind,
        "fold": fold,
        "fit_people": sorted(fit_part.person.unique().tolist()),
        "validation_people": sorted(hold_part.person.unique().tolist()),
        "epochs_ran": len(history.history.get("loss", [])),
        "status": "complete"
    })

    atomic_json(result_path, metrics)

    print("\nFOLD 5 TAMAMLANDI VE KAYDEDİLDİ")
    print("Accuracy:", round(metrics["accuracy"] * 100, 2), "%")
    print("Macro F1:", round(metrics["macro_f1"], 4))
    print("Balanced Accuracy:", round(metrics["balanced_accuracy"], 4))

Fold: 5
Eğitim görseli: 6358
Doğrulama görseli: 2139
Epoch 1/40
199/199 ━━━━━━━━━━━━━━━━━━━━ 35s 152ms/step - accuracy: 0.5925 - loss: 0.9881 - val_accuracy: 0.2412 - val_loss: 1.6181 - learning_rate: 0.0010
Epoch 2/40
199/199 ━━━━━━━━━━━━━━━━━━━━ 29s 146ms/step - accuracy: 0.8040 - loss: 0.5797 - val_accuracy: 0.3913 - val_loss: 1.4251 - learning_rate: 0.0010
Epoch 3/40
199/199 ━━━━━━━━━━━━━━━━━━━━ 34s 169ms/step - accuracy: 0.8838 - loss: 0.4010 - val_accuracy: 0.5185 - val_loss: 1.1974 - learning_rate: 0.0010
Epoch 4/40
199/199 ━━━━━━━━━━━━━━━━━━━━ 41s 169ms/step - accuracy: 0.9190 - loss: 0.3014 - val_accuracy: 0.7550 - val_loss: 0.6636 - learning_rate: 0.0010
Epoch 5/40
199/199 ━━━━━━━━━━━━━━━━━━━━ 41s 168ms/step - accuracy: 0.9377 - loss: 0.2452 - val_accuracy: 0.2445 - val_loss: 4.1432 - learning_rate: 0.0010
Epoch 6/40
199/199 ━━━━━━━━━━━━━━━━━━━━ 34s 168ms/step - accuracy: 0.9497 - loss: 0.2075 - val_accuracy: 0.7223 - val_loss: 0.6982 - learning_rate: 0.0010
Epoch 7/40
199/19

In [23]:
import json
import pandas as pd

results = []

for fold in range(1, 6):
    path = (
        RUN_DIR / "folds" /
        "small_cnn" /
        f"fold_{fold}" /
        "metrics.json"
    )

    with open(path, encoding="utf-8") as f:
        metrics = json.load(f)

    results.append({
        "Fold": fold,
        "Accuracy": metrics["accuracy"],
        "Macro F1": metrics["macro_f1"],
        "Balanced Accuracy": metrics["balanced_accuracy"]
    })

results_df = pd.DataFrame(results)

display(results_df)

print("\nSMALL CNN ORTALAMA SONUÇLARI")
print("Accuracy:", round(results_df["Accuracy"].mean() * 100, 2), "%")
print("Macro F1:", round(results_df["Macro F1"].mean(), 4))
print("Balanced Accuracy:", round(results_df["Balanced Accuracy"].mean(), 4))

,Fold,Accuracy,Macro F1,Balanced Accuracy
0,1,0.698908,0.708347,0.701230
1,2,0.930833,0.932423,0.930833
2,3,0.650655,0.648794,0.643793
3,4,0.966732,0.960807,0.964081
4,5,0.755026,0.756483,0.756613



SMALL CNN ORTALAMA SONUÇLARI
Accuracy: 80.04 %
Macro F1: 0.8014
Balanced Accuracy: 0.7993


In [24]:

# MOBILENETV3SMALL - FOLD 1

import json
import tensorflow as tf

kind = "mobilenet_v3_small"
fold = 1

fold_dir = RUN_DIR / "folds" / kind / f"fold_{fold}"
fold_dir.mkdir(parents=True, exist_ok=True)

result_path = fold_dir / "metrics.json"

if result_path.exists():
    print("MobileNetV3Small Fold 1 zaten tamamlanmış.")
else:
    cv_splits = list(
        cv.split(
            train_df,
            train_df.label,
            groups=train_df.person
        )
    )

    fit_idx, hold_idx = cv_splits[fold - 1]
    fit_part = train_df.iloc[fit_idx].copy()
    hold_part = train_df.iloc[hold_idx].copy()

    print("Model:", kind)
    print("Fold:", fold)
    print("Eğitim görseli:", len(fit_part))
    print("Doğrulama görseli:", len(hold_part))

    # Ön eğitimli tabanı dondurulmuş MobileNetV3Small
    model = build_model(kind)

    print("\nMobileNetV3Small Fold 1 eğitimi başlıyor...")

    history = model.fit(
        make_ds(fit_part, training=True),
        validation_data=make_ds(hold_part),
        epochs=MAX_EPOCHS,
        class_weight=class_weights(fit_part),
        callbacks=callbacks_for(fold_dir),
        verbose=1
    )

    # En iyi checkpoint'i yükle
    best_model = tf.keras.models.load_model(
        fold_dir / "best.keras",
        compile=True
    )

    # Doğrulama sonuçlarını hesapla
    metrics = evaluate_model(
        best_model,
        hold_part,
        "cv_mobilenet_v3_small_fold_1",
        fold_dir
    )

    metrics.update({
        "model": kind,
        "fold": fold,
        "fit_people": sorted(fit_part.person.unique().tolist()),
        "validation_people": sorted(
            hold_part.person.unique().tolist()
        ),
        "epochs_ran": len(json.loads(
            (fold_dir / "history.json").read_text(
                encoding="utf-8"
            )
        )),
        "status": "complete"
    })

    atomic_json(result_path, metrics)

    print("\nMOBILENETV3SMALL FOLD 1 TAMAMLANDI")
    print("Accuracy:", round(metrics["accuracy"] * 100, 2), "%")
    print("Macro F1:", round(metrics["macro_f1"], 4))
    print(
        "Balanced Accuracy:",
        round(metrics["balanced_accuracy"], 4)
    )
    print("Kayıt:", result_path)


Model: mobilenet_v3_small
Fold: 1
Eğitim görseli: 7215
Doğrulama görseli: 1282


/usr/local/lib/python3.13/dist-packages/keras/src/applications/mobilenet_v3.py:454: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

MobileNetV3Small Fold 1 eğitimi başlıyor...
Epoch 1/40
226/226 ━━━━━━━━━━━━━━━━━━━━ 39s 143ms/step - accuracy: 0.8279 - loss: 0.5301 - val_accuracy: 0.7902 - val_loss: 0.5152 - learning_rate: 0.0010
Epoch 2/40
226/226 ━━━━━━━━━━━━━━━━━━━━ 40s 142ms/step - accuracy: 0.9727 - loss: 0.1475 - val_accuracy: 0.8229 - val_loss: 0.4063 - learning_rate: 0.0010
Epoch 3/40
226/226 ━━━━━━━━━━━━━━━━━━━━ 42s 148ms/step - accuracy: 0.9810 - loss: 0.1009 - val_accuracy: 0.8120 - val_loss: 0.3897 - learning_rate: 0.0010
Epoch 4/40
226/226 ━━━━━━━━━━━━━━━━━━━━ 31s 136ms/step - accuracy: 0.9856 - loss: 0.0736 - val_accuracy: 0.8791 - val_loss: 0.3106 - learning_rate: 0.0010
Epoch 5/40
226/226 ━━━━━━━━━━━━━━━━━━━━ 31s 137ms/step - accuracy: 0.9885 - loss: 0.0586 - val_accuracy: 0.8502 - val_loss: 0.3477 - learning_rate: 0.0010
Epoch 6/40
226/226 ━━━━━━━━━━━━━━━━━━━━ 31s 137ms/step - accuracy: 0.9900 - loss: 0.0517 - val_accuracy: 0.8456 - val_loss: 0.3250 

In [25]:

# MOBILENETV3SMALL - FOLD 2

import json
import tensorflow as tf

kind = "mobilenet_v3_small"
fold = 2

fold_dir = RUN_DIR / "folds" / kind / f"fold_{fold}"
fold_dir.mkdir(parents=True, exist_ok=True)

result_path = fold_dir / "metrics.json"

if result_path.exists():
    print("Fold 2 zaten tamamlanmış.")
else:
    cv_splits = list(
        cv.split(
            train_df,
            train_df.label,
            groups=train_df.person
        )
    )

    fit_idx, hold_idx = cv_splits[fold - 1]
    fit_part = train_df.iloc[fit_idx].copy()
    hold_part = train_df.iloc[hold_idx].copy()

    print("Model:", kind)
    print("Fold:", fold)
    print("Eğitim görseli:", len(fit_part))
    print("Doğrulama görseli:", len(hold_part))

    model = build_model(kind)

    history = model.fit(
        make_ds(fit_part, training=True),
        validation_data=make_ds(hold_part),
        epochs=MAX_EPOCHS,
        class_weight=class_weights(fit_part),
        callbacks=callbacks_for(fold_dir),
        verbose=1
    )

    best_model = tf.keras.models.load_model(
        fold_dir / "best.keras",
        compile=True
    )

    metrics = evaluate_model(
        best_model,
        hold_part,
        "cv_mobilenet_v3_small_fold_2",
        fold_dir
    )

    metrics.update({
        "model": kind,
        "fold": fold,
        "fit_people": sorted(fit_part.person.unique().tolist()),
        "validation_people": sorted(
            hold_part.person.unique().tolist()
        ),
        "epochs_ran": len(json.loads(
            (fold_dir / "history.json").read_text(
                encoding="utf-8"
            )
        )),
        "status": "complete"
    })

    atomic_json(result_path, metrics)

    print("\nMOBILENETV3SMALL FOLD 2 TAMAMLANDI")
    print("Accuracy:", round(metrics["accuracy"] * 100, 2), "%")
    print("Macro F1:", round(metrics["macro_f1"], 4))
    print(
        "Balanced Accuracy:",
        round(metrics["balanced_accuracy"], 4)
    )
    print("Kayıt:", result_path)


Model: mobilenet_v3_small
Fold: 2
Eğitim görseli: 7297
Doğrulama görseli: 1200


/usr/local/lib/python3.13/dist-packages/keras/src/applications/mobilenet_v3.py:454: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


Epoch 1/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 40s 145ms/step - accuracy: 0.8191 - loss: 0.5406 - val_accuracy: 1.0000 - val_loss: 0.1125 - learning_rate: 0.0010
Epoch 2/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 31s 136ms/step - accuracy: 0.9700 - loss: 0.1597 - val_accuracy: 1.0000 - val_loss: 0.0417 - learning_rate: 0.0010
Epoch 3/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 31s 137ms/step - accuracy: 0.9770 - loss: 0.1074 - val_accuracy: 1.0000 - val_loss: 0.0270 - learning_rate: 0.0010
Epoch 4/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 31s 135ms/step - accuracy: 0.9831 - loss: 0.0833 - val_accuracy: 1.0000 - val_loss: 0.0147 - learning_rate: 0.0010
Epoch 5/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 31s 137ms/step - accuracy: 0.9830 - loss: 0.0734 - val_accuracy: 1.0000 - val_loss: 0.0111 - learning_rate: 0.0010
Epoch 6/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 41s 136ms/step - accuracy: 0.9874 - loss: 0.0573 - val_accuracy: 1.0000 - val_loss: 0.0091 - learning_rate: 0.0010
Epoch 7/40
229/229 ━━━━━━━━━━━━━━━━━━━━ 31s 135ms/step - accuracy: 0.9

In [26]:

# MOBILENETV3SMALL - FOLD 3

import json
import tensorflow as tf

kind = "mobilenet_v3_small"
fold = 3

fold_dir = RUN_DIR / "folds" / kind / f"fold_{fold}"
fold_dir.mkdir(parents=True, exist_ok=True)

result_path = fold_dir / "metrics.json"

if result_path.exists():
    print("Fold 3 zaten tamamlanmış.")
else:
    cv_splits = list(
        cv.split(
            train_df,
            train_df.label,
            groups=train_df.person
        )
    )

    fit_idx, hold_idx = cv_splits[fold - 1]

    fit_part = train_df.iloc[fit_idx].copy()
    hold_part = train_df.iloc[hold_idx].copy()

    print("Model:", kind)
    print("Fold:", fold)
    print("Eğitim görseli:", len(fit_part))
    print("Doğrulama görseli:", len(hold_part))

    model = build_model(kind)

    history = model.fit(
        make_ds(fit_part, training=True),
        validation_data=make_ds(hold_part),
        epochs=MAX_EPOCHS,
        class_weight=class_weights(fit_part),
        callbacks=callbacks_for(fold_dir),
        verbose=1
    )

    best_model = tf.keras.models.load_model(
        fold_dir / "best.keras",
        compile=True
    )

    metrics = evaluate_model(
        best_model,
        hold_part,
        "cv_mobilenet_v3_small_fold_3",
        fold_dir
    )

    metrics.update({
        "model": kind,
        "fold": fold,
        "fit_people": sorted(
            fit_part.person.unique().tolist()
        ),
        "validation_people": sorted(
            hold_part.person.unique().tolist()
        ),
        "epochs_ran": len(json.loads(
            (fold_dir / "history.json").read_text(
                encoding="utf-8"
            )
        )),
        "status": "complete"
    })

    atomic_json(result_path, metrics)

    print("\nMOBILENETV3SMALL FOLD 3 TAMAMLANDI")
    print("Accuracy:", round(metrics["accuracy"] * 100, 2), "%")
    print("Macro F1:", round(metrics["macro_f1"], 4))
    print(
        "Balanced Accuracy:",
        round(metrics["balanced_accuracy"], 4)
    )
    print("Kayıt:", result_path)


Model: mobilenet_v3_small
Fold: 3
Eğitim görseli: 6665
Doğrulama görseli: 1832


/usr/local/lib/python3.13/dist-packages/keras/src/applications/mobilenet_v3.py:454: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


Epoch 1/40
209/209 ━━━━━━━━━━━━━━━━━━━━ 38s 152ms/step - accuracy: 0.8290 - loss: 0.5236 - val_accuracy: 0.9727 - val_loss: 0.1799 - learning_rate: 0.0010
Epoch 2/40
209/209 ━━━━━━━━━━━━━━━━━━━━ 30s 142ms/step - accuracy: 0.9739 - loss: 0.1478 - val_accuracy: 0.9738 - val_loss: 0.1199 - learning_rate: 0.0010
Epoch 3/40
209/209 ━━━━━━━━━━━━━━━━━━━━ 30s 142ms/step - accuracy: 0.9835 - loss: 0.0903 - val_accuracy: 0.9760 - val_loss: 0.0987 - learning_rate: 0.0010
Epoch 4/40
209/209 ━━━━━━━━━━━━━━━━━━━━ 41s 142ms/step - accuracy: 0.9857 - loss: 0.0722 - val_accuracy: 0.9760 - val_loss: 0.0870 - learning_rate: 0.0010
Epoch 5/40
209/209 ━━━━━━━━━━━━━━━━━━━━ 42s 147ms/step - accuracy: 0.9895 - loss: 0.0546 - val_accuracy: 0.9743 - val_loss: 0.0847 - learning_rate: 0.0010
Epoch 6/40
209/209 ━━━━━━━━━━━━━━━━━━━━ 30s 144ms/step - accuracy: 0.9895 - loss: 0.0486 - val_accuracy: 0.9749 - val_loss: 0.0789 - learning_rate: 0.0010
Epoch 7/40
209/209 ━━━━━━━━━━━━━━━━━━━━ 40s 138ms/step - accuracy: 0.9

In [27]:

# MOBILENETV3SMALL - FOLD 4

import json
import tensorflow as tf

kind = "mobilenet_v3_small"
fold = 4

fold_dir = RUN_DIR / "folds" / kind / f"fold_{fold}"
fold_dir.mkdir(parents=True, exist_ok=True)

result_path = fold_dir / "metrics.json"

if result_path.exists():
    print("Fold 4 zaten tamamlanmış.")
else:
    cv_splits = list(
        cv.split(
            train_df,
            train_df.label,
            groups=train_df.person
        )
    )

    fit_idx, hold_idx = cv_splits[fold - 1]

    fit_part = train_df.iloc[fit_idx].copy()
    hold_part = train_df.iloc[hold_idx].copy()

    print("Model:", kind)
    print("Fold:", fold)
    print("Eğitim görseli:", len(fit_part))
    print("Doğrulama görseli:", len(hold_part))

    model = build_model(kind)

    history = model.fit(
        make_ds(fit_part, training=True),
        validation_data=make_ds(hold_part),
        epochs=MAX_EPOCHS,
        class_weight=class_weights(fit_part),
        callbacks=callbacks_for(fold_dir),
        verbose=1
    )

    best_model = tf.keras.models.load_model(
        fold_dir / "best.keras",
        compile=True
    )

    metrics = evaluate_model(
        best_model,
        hold_part,
        "cv_mobilenet_v3_small_fold_4",
        fold_dir
    )

    metrics.update({
        "model": kind,
        "fold": fold,
        "fit_people": sorted(fit_part.person.unique().tolist()),
        "validation_people": sorted(
            hold_part.person.unique().tolist()
        ),
        "epochs_ran": len(json.loads(
            (fold_dir / "history.json").read_text(
                encoding="utf-8"
            )
        )),
        "status": "complete"
    })

    atomic_json(result_path, metrics)

    print("\nMOBILENETV3SMALL FOLD 4 TAMAMLANDI")
    print("Accuracy:", round(metrics["accuracy"] * 100, 2), "%")
    print("Macro F1:", round(metrics["macro_f1"], 4))
    print(
        "Balanced Accuracy:",
        round(metrics["balanced_accuracy"], 4)
    )
    print("Kayıt:", result_path)


Model: mobilenet_v3_small
Fold: 4
Eğitim görseli: 6453
Doğrulama görseli: 2044


/usr/local/lib/python3.13/dist-packages/keras/src/applications/mobilenet_v3.py:454: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


Epoch 1/40
202/202 ━━━━━━━━━━━━━━━━━━━━ 38s 150ms/step - accuracy: 0.8639 - loss: 0.4527 - val_accuracy: 0.8596 - val_loss: 0.4151 - learning_rate: 0.0010
Epoch 2/40
202/202 ━━━━━━━━━━━━━━━━━━━━ 34s 169ms/step - accuracy: 0.9797 - loss: 0.1266 - val_accuracy: 0.8943 - val_loss: 0.3064 - learning_rate: 0.0010
Epoch 3/40
202/202 ━━━━━━━━━━━━━━━━━━━━ 41s 170ms/step - accuracy: 0.9873 - loss: 0.0826 - val_accuracy: 0.9159 - val_loss: 0.2492 - learning_rate: 0.0010
Epoch 4/40
202/202 ━━━━━━━━━━━━━━━━━━━━ 37s 149ms/step - accuracy: 0.9887 - loss: 0.0620 - val_accuracy: 0.9310 - val_loss: 0.2180 - learning_rate: 0.0010
Epoch 5/40
202/202 ━━━━━━━━━━━━━━━━━━━━ 45s 171ms/step - accuracy: 0.9895 - loss: 0.0517 - val_accuracy: 0.9428 - val_loss: 0.1952 - learning_rate: 0.0010
Epoch 6/40
202/202 ━━━━━━━━━━━━━━━━━━━━ 35s 173ms/step - accuracy: 0.9918 - loss: 0.0429 - val_accuracy: 0.9423 - val_loss: 0.1938 - learning_rate: 0.0010
Epoch 7/40
202/202 ━━━━━━━━━━━━━━━━━━━━ 30s 147ms/step - accuracy: 0.9

In [28]:

# MOBILENETV3SMALL - FOLD 5

import json
import tensorflow as tf

kind = "mobilenet_v3_small"
fold = 5

fold_dir = RUN_DIR / "folds" / kind / f"fold_{fold}"
fold_dir.mkdir(parents=True, exist_ok=True)

result_path = fold_dir / "metrics.json"

if result_path.exists():
    print("Fold 5 zaten tamamlanmış.")
else:
    cv_splits = list(
        cv.split(
            train_df,
            train_df.label,
            groups=train_df.person
        )
    )

    fit_idx, hold_idx = cv_splits[fold - 1]

    fit_part = train_df.iloc[fit_idx].copy()
    hold_part = train_df.iloc[hold_idx].copy()

    print("Model:", kind)
    print("Fold:", fold)
    print("Eğitim görseli:", len(fit_part))
    print("Doğrulama görseli:", len(hold_part))

    model = build_model(kind)

    history = model.fit(
        make_ds(fit_part, training=True),
        validation_data=make_ds(hold_part),
        epochs=MAX_EPOCHS,
        class_weight=class_weights(fit_part),
        callbacks=callbacks_for(fold_dir),
        verbose=1
    )

    best_model = tf.keras.models.load_model(
        fold_dir / "best.keras",
        compile=True
    )

    metrics = evaluate_model(
        best_model,
        hold_part,
        "cv_mobilenet_v3_small_fold_5",
        fold_dir
    )

    metrics.update({
        "model": kind,
        "fold": fold,
        "fit_people": sorted(
            fit_part.person.unique().tolist()
        ),
        "validation_people": sorted(
            hold_part.person.unique().tolist()
        ),
        "epochs_ran": len(json.loads(
            (fold_dir / "history.json").read_text(
                encoding="utf-8"
            )
        )),
        "status": "complete"
    })

    atomic_json(result_path, metrics)

    print("\nMOBILENETV3SMALL FOLD 5 TAMAMLANDI")
    print("Accuracy:", round(metrics["accuracy"] * 100, 2), "%")
    print("Macro F1:", round(metrics["macro_f1"], 4))
    print(
        "Balanced Accuracy:",
        round(metrics["balanced_accuracy"], 4)
    )
    print("Kayıt:", result_path)


Model: mobilenet_v3_small
Fold: 5
Eğitim görseli: 6358
Doğrulama görseli: 2139


/usr/local/lib/python3.13/dist-packages/keras/src/applications/mobilenet_v3.py:454: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


Epoch 1/40
199/199 ━━━━━━━━━━━━━━━━━━━━ 42s 176ms/step - accuracy: 0.7941 - loss: 0.5941 - val_accuracy: 0.9804 - val_loss: 0.2244 - learning_rate: 0.0010
Epoch 2/40
199/199 ━━━━━━━━━━━━━━━━━━━━ 36s 149ms/step - accuracy: 0.9707 - loss: 0.1620 - val_accuracy: 0.9794 - val_loss: 0.1344 - learning_rate: 0.0010
Epoch 3/40
199/199 ━━━━━━━━━━━━━━━━━━━━ 34s 171ms/step - accuracy: 0.9769 - loss: 0.1029 - val_accuracy: 0.9808 - val_loss: 0.1038 - learning_rate: 0.0010
Epoch 4/40
199/199 ━━━━━━━━━━━━━━━━━━━━ 37s 149ms/step - accuracy: 0.9851 - loss: 0.0797 - val_accuracy: 0.9818 - val_loss: 0.0854 - learning_rate: 0.0010
Epoch 5/40
199/199 ━━━━━━━━━━━━━━━━━━━━ 41s 151ms/step - accuracy: 0.9885 - loss: 0.0636 - val_accuracy: 0.9813 - val_loss: 0.0811 - learning_rate: 0.0010
Epoch 6/40
199/199 ━━━━━━━━━━━━━━━━━━━━ 44s 166ms/step - accuracy: 0.9885 - loss: 0.0546 - val_accuracy: 0.9790 - val_loss: 0.0870 - learning_rate: 0.0010
Epoch 7/40
199/199 ━━━━━━━━━━━━━━━━━━━━ 28s 142ms/step - accuracy: 0.9

In [29]:

import json
import pandas as pd

cv_results = []

for kind in ["small_cnn", "mobilenet_v3_small"]:
    for fold in range(1, 6):
        path = RUN_DIR / "folds" / kind / f"fold_{fold}" / "metrics.json"

        if not path.exists():
            raise FileNotFoundError(f"Eksik fold sonucu: {path}")

        with open(path, encoding="utf-8") as f:
            result = json.load(f)

        cv_results.append(result)

cv_df = pd.DataFrame(cv_results)

summary = cv_df.groupby("model").agg(
    folds=("fold", "count"),
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_std=("macro_f1", "std"),
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
    weighted_f1_mean=("weighted_f1", "mean")
).sort_values("macro_f1_mean", ascending=False)

display(summary)

selected_model = summary.index[0]

atomic_json(
    RUN_DIR / "reports" / "cv_model_comparison.json",
    {
        "selected_model": selected_model,
        "summary": summary.reset_index().to_dict("records"),
        "fold_results": cv_results
    }
)

print("\nCV'YE GÖRE SEÇİLEN MİMARİ:", selected_model)


,folds,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,weighted_f1_mean
model,,,,,
mobilenet_v3_small,5,0.956665,0.053589,0.957469,0.958584
small_cnn,5,0.801371,0.138332,0.799310,0.805537



CV'YE GÖRE SEÇİLEN MİMARİ: mobilenet_v3_small


In [35]:

from pathlib import Path

files_to_check = {
    "Final model": RUN_DIR / "checkpoints" / "final" / "final_model.keras",
    "Head modeli": RUN_DIR / "checkpoints" / "final" / "head" / "best.keras",
    "Fine-tuning modeli": RUN_DIR / "checkpoints" / "final" / "finetune" / "best.keras",
    "Final test metrikleri": RUN_DIR / "reports" / "final_test_metrics.json",
    "Model karşılaştırması": RUN_DIR / "reports" / "cv_model_comparison.json",
    "Eğitim aşamaları": RUN_DIR / "checkpoints" / "final" / "stage_summaries.json",
}

for kind in ["small_cnn", "mobilenet_v3_small"]:
    for fold in range(1, 6):
        files_to_check[f"{kind} Fold {fold}"] = (
            RUN_DIR / "folds" / kind / f"fold_{fold}" / "metrics.json"
        )

missing = []

for name, path in files_to_check.items():
    if path.exists():
        print("OK   ", name)
    else:
        print("EKSIK", name)
        missing.append(name)

print("\nKontrol edilen:", len(files_to_check))
print("Eksik:", len(missing))


OK    Final model
OK    Head modeli
OK    Fine-tuning modeli
OK    Final test metrikleri
OK    Model karşılaştırması
OK    Eğitim aşamaları
OK    small_cnn Fold 1
OK    small_cnn Fold 2
OK    small_cnn Fold 3
OK    small_cnn Fold 4
OK    small_cnn Fold 5
OK    mobilenet_v3_small Fold 1
OK    mobilenet_v3_small Fold 2
OK    mobilenet_v3_small Fold 3
OK    mobilenet_v3_small Fold 4
OK    mobilenet_v3_small Fold 5

Kontrol edilen: 16
Eksik: 0
